In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

import joblib
import mlflow
import mlflow.sklearn

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "heart_disease_processed.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [4]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (242, 13)
Testing features: (61, 13)
Training target: (242,)
Testing target: (61,)


In [5]:
numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [6]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

logistic_pipeline.fit(X_train, y_train)

log_pred = logistic_pipeline.predict(X_test)
log_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Logistic Regression Accuracy:", round(accuracy_score(y_test, log_pred), 4))
print("Logistic Regression Precision:", round(precision_score(y_test, log_pred), 4))
print("Logistic Regression Recall:", round(recall_score(y_test, log_pred), 4))
print("Logistic Regression ROC-AUC:", round(roc_auc_score(y_test, log_prob), 4))

Logistic Regression Accuracy: 0.8852
Logistic Regression Precision: 0.8387
Logistic Regression Recall: 0.9286
Logistic Regression ROC-AUC: 0.9665


In [9]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42
            )
        )
    ]
)

random_forest_pipeline.fit(X_train, y_train)

rf_predictions = random_forest_pipeline.predict(X_test)
rf_probabilities = random_forest_pipeline.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions)
rf_recall = recall_score(y_test, rf_predictions)
rf_roc_auc = roc_auc_score(y_test, rf_probabilities)

print("Random Forest Results")
print("---------------------")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("ROC-AUC  :", round(rf_roc_auc, 4))

Random Forest Results
---------------------
Accuracy : 0.8525
Precision: 0.8065
Recall   : 0.8929
ROC-AUC  : 0.9432


In [10]:
from sklearn.model_selection import cross_validate

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc"
}

models = {
    "Logistic Regression": logistic_pipeline,
    "Random Forest": random_forest_pipeline
}

cv_results = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=5,
        scoring=scoring
    )

    cv_results.append({
        "Model": model_name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_results_df = pd.DataFrame(cv_results)

print(cv_results_df.round(4).to_string(index=False))

              Model  CV Accuracy  CV Precision  CV Recall  CV ROC-AUC
Logistic Regression       0.8428        0.8686     0.7739      0.8937
      Random Forest       0.8055        0.8050     0.7648      0.8983


In [11]:
from sklearn.model_selection import GridSearchCV

logistic_params = {
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__class_weight": [None, "balanced"]
}

logistic_grid = GridSearchCV(
    logistic_pipeline,
    logistic_params,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

logistic_grid.fit(X_train, y_train)

rf_params = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 5, 10],
    "classifier__min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    random_forest_pipeline,
    rf_params,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Logistic Regression")
print("Best Parameters:", logistic_grid.best_params_)
print("Best CV ROC-AUC:", round(logistic_grid.best_score_, 4))

print("\nRandom Forest")
print("Best Parameters:", rf_grid.best_params_)
print("Best CV ROC-AUC:", round(rf_grid.best_score_, 4))

Logistic Regression
Best Parameters: {'classifier__C': 0.1, 'classifier__class_weight': None}
Best CV ROC-AUC: 0.8982

Random Forest
Best Parameters: {'classifier__max_depth': 5, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}
Best CV ROC-AUC: 0.9043


In [13]:
best_logistic = logistic_grid.best_estimator_
best_rf = rf_grid.best_estimator_

log_pred = best_logistic.predict(X_test)
log_prob = best_logistic.predict_proba(X_test)[:, 1]

rf_pred = best_rf.predict(X_test)
rf_prob = best_rf.predict_proba(X_test)[:, 1]

final_results = pd.DataFrame({
    "Model": [
        "Tuned Logistic Regression",
        "Tuned Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, log_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Precision": [
        precision_score(y_test, log_pred),
        precision_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, log_pred),
        recall_score(y_test, rf_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, log_prob),
        roc_auc_score(y_test, rf_prob)
    ]
})

print(final_results.round(4).to_string(index=False))

                    Model  Accuracy  Precision  Recall  ROC-AUC
Tuned Logistic Regression    0.8852     0.8387  0.9286   0.9654
      Tuned Random Forest    0.8525     0.8276  0.8571   0.9470


In [14]:
import joblib

final_model = best_logistic

results_path = PROJECT_ROOT / "screenshots" / "model_comparison.csv"
final_results.round(4).to_csv(results_path, index=False)

model_path = PROJECT_ROOT / "models" / "heart_disease_pipeline.joblib"
joblib.dump(final_model, model_path)

print("Final model: Tuned Logistic Regression")
print("Model saved to:", model_path)
print("Comparison saved to:", results_path)

Final model: Tuned Logistic Regression
Model saved to: /home/s2025ae05561/workspace/heart-disease-mlops/models/heart_disease_pipeline.joblib
Comparison saved to: /home/s2025ae05561/workspace/heart-disease-mlops/screenshots/model_comparison.csv


In [15]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("heart-disease-classification")

with mlflow.start_run(run_name="Tuned_Logistic_Regression"):

    mlflow.log_param("model", "Logistic Regression")
    mlflow.log_param("C", logistic_grid.best_params_["classifier__C"])
    mlflow.log_param(
        "class_weight",
        str(logistic_grid.best_params_["classifier__class_weight"])
    )
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("accuracy", accuracy_score(y_test, log_pred))
    mlflow.log_metric("precision", precision_score(y_test, log_pred))
    mlflow.log_metric("recall", recall_score(y_test, log_pred))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, log_prob))
    mlflow.log_metric("best_cv_roc_auc", logistic_grid.best_score_)

    mlflow.log_artifact(str(results_path))

    mlflow.sklearn.log_model(
        final_model,
        name="heart_disease_model",
        serialization_format="cloudpickle"
    )

print("MLflow Logistic Regression run logged successfully.")

2026/10/06 15:15:31 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/06 15:15:32 INFO mlflow.store.db.utils: Updating database tables
2026/10/06 15:16:04 INFO mlflow.tracking.fluent: Experiment with name 'heart-disease-classification' does not exist. Creating a new experiment.
2026/10/06 15:16:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MLflow Logistic Regression run logged successfully.


In [16]:
with mlflow.start_run(run_name="Tuned_Random_Forest"):

    mlflow.log_param("model", "Random Forest")
    mlflow.log_param(
        "n_estimators",
        rf_grid.best_params_["classifier__n_estimators"]
    )
    mlflow.log_param(
        "max_depth",
        str(rf_grid.best_params_["classifier__max_depth"])
    )
    mlflow.log_param(
        "min_samples_split",
        rf_grid.best_params_["classifier__min_samples_split"]
    )
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("accuracy", accuracy_score(y_test, rf_pred))
    mlflow.log_metric("precision", precision_score(y_test, rf_pred))
    mlflow.log_metric("recall", recall_score(y_test, rf_pred))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, rf_prob))
    mlflow.log_metric("best_cv_roc_auc", rf_grid.best_score_)

    mlflow.log_artifact(str(results_path))

    mlflow.sklearn.log_model(
        best_rf,
        name="heart_disease_model",
        serialization_format="cloudpickle"
    )

print("MLflow Random Forest run logged successfully.")

2026/10/06 15:17:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MLflow Random Forest run logged successfully.
